# mcp_filesystem_client: Using someone else's MCP server

**This folder has no server of its own. The client starts the official filesystem server, a Node.js program published on npm, and uses its tools to write, read, move and search files inside one folder, `sandbox/`.** That's the point of MCP: Any client can use any server, whoever wrote it and in whatever language.

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
 this notebook (Python)           server-filesystem (Node.js)
 ----------------------           ---------------------------
 call_tool("write_file") -stdin-&gt;  writes, inside sandbox/ only
 reads the result       &lt;-stdout-  "Successfully wrote to ..."
</pre>

Read `../crash-course/2-simple-server-setup/client-stdio.ipynb` first. The connection is the same.

## 1. How the server gets started

- **`npx`:** A tool that comes with Node.js. `npx -y <package>` downloads a package from npm the first time (then it's cached) and runs it. `-y` answers "yes" to the download question.
- **`@2.0.1`:** Pins the version. Without it, `npx` fetches the newest version, so the tools could change under you.
- **The last argument, `sandbox`:** The only folder the server may touch. It refuses any path outside it (section 4).
- **Why `npx` and not `python`:** `command` can be any program. stdio only needs a program that reads JSON messages from its stdin and writes them to its stdout.

In [1]:
import os

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

sandbox = os.path.abspath("sandbox")
server = StdioServerParameters(
    command="npx",
    args=["-y", "@modelcontextprotocol/server-filesystem@2.0.1", sandbox],
)
print(server.command)
print(server.args)

npx
['-y', '@modelcontextprotocol/server-filesystem@2.0.1', 'C:\\Users\\ashut\\ML_Practice\\MCP\\mcp_filesystem_client\\sandbox']


## 2. Writing, reading, moving and searching

`os.path.abspath("sandbox")` turns the folder name into a full path. The server works with full paths, and so does each call below. Before connecting, the cell removes the moved file from any earlier run, because `move_file` refuses to overwrite.

In [2]:
os.makedirs(sandbox, exist_ok=True)
file_path = os.path.join(sandbox, "hello.txt")
sub_dir = os.path.join(sandbox, "subdir")
new_path = os.path.join(sub_dir, "hello_moved.txt")
outside_path = os.path.abspath("client.py")

if os.path.exists(new_path):
    os.remove(new_path)

with open("server_log.txt", "w") as log:
    async with stdio_client(server, errlog=log) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools_result = await session.list_tools()
            print(len(tools_result.tools), "tools")

            result = await session.call_tool("write_file", {"path": file_path, "content": "Hello from my MCP client!"})
            print(result.content[0].text)

            result = await session.call_tool("read_text_file", {"path": file_path})
            print(result.content[0].text)

            result = await session.call_tool("create_directory", {"path": sub_dir})
            print(result.content[0].text)

            result = await session.call_tool("move_file", {"source": file_path, "destination": new_path})
            print(result.content[0].text)

            result = await session.call_tool("search_files", {"path": sandbox, "pattern": "**/*.txt"})
            print(result.content[0].text)

            outside = await session.call_tool("read_text_file", {"path": outside_path})

14 tools
Successfully wrote to C:\Users\ashut\ML_Practice\MCP\mcp_filesystem_client\sandbox\hello.txt
Hello from my MCP client!
Successfully created directory C:\Users\ashut\ML_Practice\MCP\mcp_filesystem_client\sandbox\subdir
Successfully moved C:\Users\ashut\ML_Practice\MCP\mcp_filesystem_client\sandbox\hello.txt to C:\Users\ashut\ML_Practice\MCP\mcp_filesystem_client\sandbox\subdir\hello_moved.txt
C:\Users\ashut\ML_Practice\MCP\mcp_filesystem_client\sandbox\subdir\hello_moved.txt


Each tool answered with one text block:

- **`write_file`:** Created `hello.txt` inside the sandbox.
- **`read_text_file`:** Returned its contents. Its older name, `read_file`, still works but is marked deprecated.
- **`create_directory`:** Made `subdir`.
- **`move_file`:** Moved the file into `subdir`.
- **`search_files`:** Found it. `**/*.txt` means "any `.txt` file in any subfolder". A plain `*.txt` only matches the top folder.

## 3. Its tools

The server offers 14 tools. These are their names:

In [3]:
for tool in tools_result.tools:
    print(tool.name)

read_file
read_text_file
read_media_file
read_multiple_files
write_file
edit_file
create_directory
list_directory
list_directory_with_sizes
directory_tree
move_file
search_files
get_file_info
list_allowed_directories


Notice what's missing: There's no tool to delete a file. The server's authors chose what an LLM may do, and deleting isn't on the list.

## 4. The sandbox protects your files

The last call asked for `client.py`, which is outside the sandbox:

In [4]:
print(outside.isError)
print(outside.content[0].text)

True
Access denied - path outside allowed directories: C:\Users\ashut\ML_Practice\MCP\mcp_filesystem_client\client.py not in C:\Users\ashut\ML_Practice\MCP\mcp_filesystem_client\sandbox


The server refused it. The check happens in the server, not in this client, so it holds even if a model asks for the wrong path. That's the security idea behind it: Give each server only the folder it needs.

The server's log:

In [5]:
with open("server_log.txt") as log:
    print(log.read())

Secure MCP Filesystem Server running on stdio
Client does not support MCP Roots, using allowed directories set from server args: [
  'C:\\Users\\ashut\\ML_Practice\\MCP\\mcp_filesystem_client\\sandbox'
]



"Client does not support MCP Roots" means this client didn't send a list of allowed folders of its own, so the server used the folder from its command line. Roots are deprecated in the newest spec, which says to pass folders as server arguments, as here.

On your machine, the first run may also print an npm warning that this version wants Node 24 (you have Node 20). It still runs. Upgrading Node to 24 removes the warning.

## 5. The .py version

`client.py` does the same, except that it prints the refusal straight away. Run from this folder with `python client.py`:

<div style="font-size: 0.85em">

```text
14 tools
Successfully wrote to C:\Users\ashut\ML_Practice\MCP\mcp_filesystem_client\sandbox\hello.txt
Hello from my MCP client!
Successfully created directory C:\Users\ashut\ML_Practice\MCP\mcp_filesystem_client\sandbox\subdir
Successfully moved C:\Users\ashut\ML_Practice\MCP\mcp_filesystem_client\sandbox\hello.txt to C:\Users\ashut\ML_Practice\MCP\mcp_filesystem_client\sandbox\subdir\hello_moved.txt
C:\Users\ashut\ML_Practice\MCP\mcp_filesystem_client\sandbox\subdir\hello_moved.txt
True
Access denied - path outside allowed directories: C:\Users\ashut\ML_Practice\MCP\mcp_filesystem_client\client.py not in C:\Users\ashut\ML_Practice\MCP\mcp_filesystem_client\sandbox
```

</div>

## Say it out loud

> "My filesystem client has no server of its own. It starts the official Node.js filesystem server with `npx`, as a stdio child process, and passes it one allowed folder. Then it calls its tools: Write, read, move and search. The server enforces the sandbox itself. When I asked for a file outside the folder, it returned an error, 'Access denied'. That shows MCP's point: My Python client used a server written in another language by someone else, just by speaking the protocol."